# Workshop 3 - Reinforcement Learning on CartPole (Google Colab runner)

**Course:** Machine Learning - Semester 2026-III
**Workshop 3:** Reinforcement Learning (Part B) - Group 5

## Overview
This notebook runs the complete Workshop 3 pipeline, in order, from a fresh Colab
runtime. Every step calls the scripts of `workshop3/src/`, so the notebook and the
repository always produce the same numbers.

| Step | Script | What it produces |
|---|---|---|
| 1 | `env.py` | environment checks + 100 random-policy episodes |
| 2 | `w1_bridge.py` | Workshop 1 classifier rebuilt and checked |
| 3 | `train.py --sweep` | 6 configurations x 3 seeds, `checkpoints/agent_best.pt` |
| 4 | `evaluate.py` | test results of the agent and the baselines |
| 5 | `make_report_assets.py` | figures and LaTeX tables for the report |

**Runtime:** a CPU runtime is enough (the Q-network has about 17 thousand parameters).
With the stored training runs the whole notebook takes about 5 minutes; retraining
(optional, step 3) adds 20 to 40 minutes.

## 0. Get the repository

Clone the repository into `cartpole-ml` and move into it. If the runtime restarts,
run this cell again.

In [ ]:
%cd /content
!rm -rf cartpole-ml
!git clone https://github.com/Tonyaa11/Cartpole.git cartpole-ml
%cd /content/cartpole-ml

## 0.1 Dependencies

Colab already includes PyTorch, NumPy, pandas, scikit-learn, matplotlib and
TensorBoard. We only make sure Gymnasium (the RL environment API) is installed and
print the versions, which are part of the reproducibility record.

In [ ]:
!pip install -q "gymnasium>=1.0"
import torch, gymnasium, sklearn, numpy, pandas
print("torch", torch.__version__, "| gymnasium", gymnasium.__version__, "| sklearn", sklearn.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)

## 1. Validate the environment

Before training anything, the workshop asks us to check the environment:
1. the class follows the Gymnasium API (`check_env`);
2. its physics are identical to Gymnasium's `CartPole-v1` (same seed and actions give the same states);
3. 100 episodes with a **random policy**: mean return, episode-length distribution and action frequency.

In [ ]:
!python src/w3/env.py

## 2. Rebuild the Workshop 1 classifier

The Workshop 1 Gradient Boosting model is reused in two places: as a baseline policy
(one-step look-ahead) and as a metric (share of steps it flags as dangerous). This cell
rebuilds it with the Workshop 1 pipeline and checks that the fast feature code used
inside the control loop gives exactly the same features as `src/features.py`.

In [ ]:
!python src/w3/w1_bridge.py

## 3. Hyperparameter sweep (training)

Six DQN configurations, each changing **one** factor with respect to `reference`
(see `workshop3/configs/sweep.json`), trained with seeds 42, 7 and 99:

| Config | What changes |
|---|---|
| reference | lr 1e-3, gamma 0.99, epsilon decay 0.99, no weight decay, shaping k = 1 |
| lr_1e-4 | learning rate 1e-4 |
| gamma_0.95 | discount factor 0.95 |
| eps_decay_0.995 | slower exploration decay |
| weight_decay_1e-4 | L2 regularisation |
| no_shaping | task reward only |

Every 10 episodes the greedy agent is evaluated on 10 **validation** initial states; the best
checkpoint of each run is kept. The configuration is selected with validation results only,
and its best run is copied to `checkpoints/agent_best.pt`.

**The repository already contains the 18 trained runs** (`runs/workshop3/`, `checkpoints/`), and the
report was written from them. Leave `RETRAIN = False` to reuse them (steps 4 and 5 then reproduce the
report numbers exactly). Set `RETRAIN = True` only to train everything again (about 20 to 40 minutes);
PyTorch versions differ between machines, so new runs can give somewhat different numbers and the
text of the report must then be reviewed.

In [ ]:
RETRAIN = False  # True = train the 18 runs again (overwrites runs/workshop3 and checkpoints)

if RETRAIN:
    !python src/w3/train.py --sweep
else:
    !python src/w3/train.py --select-only   # re-checks the selection from the stored logs

## 3.1 TensorBoard (optional)

Per-episode return, episode length, epsilon, TD loss and validation return of every run.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/workshop3

## 4. Test evaluation

The selected agent and the baselines (random, alternating, angle rule, PD rule, W1 classifier with
look-ahead) are evaluated on **100 held-out initial states**, in the nominal environment and with
actuator noise. The test set is used only here.

In [ ]:
!python src/w3/evaluate.py

## 5. Figures and LaTeX tables

All figures and every number quoted in the report are generated from the saved logs.

In [ ]:
!python src/w3/make_report_assets_w3.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("runs/workshop3/figures/*.png")):
    print(f)
    display(Image(f))

## 6. Download the results

This zip contains what changed in the repository: logs, checkpoints, figures and the generated
LaTeX files. Unzip it over your local copy of the repository (it keeps the same folders) and
commit with GitHub Desktop.

In [ ]:
!zip -qr workshop3_results.zip runs/workshop3 checkpoints/workshop3
from google.colab import files
files.download("workshop3_results.zip")

## Summary for the report review

Prints the selection and the test table in plain text (useful to check the report numbers).

In [ ]:
import json
s = json.load(open("runs/workshop3/sweep_results.json"))
t = json.load(open("runs/workshop3/test_results.json"))
print("best configuration:", s["best_config"], "| run:", s["best_run"]["seed"], s["best_run"]["best_val_return"])
for r in s["table"]:
    print(f"{r['config']:>18}: best val {r['best_val_mean']:6.1f} +- {r['best_val_std']:5.1f} | final val {r['final_val_mean']:6.1f}")
for cond in ("nominal", "noisy"):
    print("\n", cond)
    for p in t["conditions"][cond]["policies"]:
        print(f"  {p['policy']:<28} {p['mean_return']:6.1f} +- {p['std_return']:5.1f} | success {p['success_rate']:.0%} | W1 danger {p['w1_danger_rate']:.2%}")